In [ ]:
image_path = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"

In [ ]:
import os
print(os.listdir(image_path))

In [ ]:
train_dir = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training"
val_dir = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing"

In [ ]:
import tensorflow as tf

train_data = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    seed=123,
    image_size=(224, 224),
    batch_size=16,
    label_mode="categorical"
)

val_data = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    seed=123,
    image_size=(224, 224),
    batch_size=16,
    label_mode="categorical"
)

In [ ]:
import albumentations as A
import numpy as np
import tensorflow as tf

# 1. Define the Medical-Grade Pipeline
transforms = A.Compose([

    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),

    A.HorizontalFlip(p=0.5),

    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=8, p=0.4),

    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.4),

    # A.GridDistortion(p=0.05),

    A.GaussNoise(p=0.05)
])

def aug_fn(image, label):
    aug_data = transforms(image=image.numpy())
    aug_img = aug_data["image"]
    return aug_img, label

def process_data(image, label):
    aug_img, label = tf.py_function(func=aug_fn, inp=[image, label], Tout=[tf.float32, tf.float32])

    aug_img.set_shape((224, 224, 3))
    label.set_shape((4,))

    # Normalize
    return aug_img / 255.0, label

def process_val_data(image, label):
    # Validation
    return image / 255.0, label

In [ ]:
# 3. Apply to your Datasets
auto_tune = tf.data.AUTOTUNE
BATCH_SIZE = 16

train_data = train_data.unbatch().map(process_data, num_parallel_calls=auto_tune)
train_data = train_data.prefetch(buffer_size=auto_tune)

val_data = val_data.unbatch().map(process_val_data, num_parallel_calls=auto_tune)
val_data = val_data.prefetch(buffer_size=auto_tune)

In [ ]:
train_data = train_data.shuffle(1000).prefetch(buffer_size=auto_tune)
val_data = val_data.prefetch(buffer_size=auto_tune)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

# 1. Start (conv block)
class ConvBNAct(layers.Layer):
    def __init__(self, filters, kernel_size, strides=1, padding="same"):
        super().__init__()
        self.conv = layers.Conv2D(filters, kernel_size, strides=strides, padding=padding)
        self.bn = layers.BatchNormalization()
        # self.act = layers.Activation("swish")
        self.act = layers.Activation("relu")

    def call(self, x, training=None):
        x = self.conv(x)
        x = self.bn(x, training=training)
        return self.act(x)


# 2. Squeeze-and-Excitation (Se) Block
class SEBlock(layers.Layer):
    def __init__(self, filters, ratio=16):
        super().__init__()
        self.pool = layers.GlobalAveragePooling2D()
        self.fc1 = layers.Dense(filters // ratio, activation="swish", kernel_initializer=...)
        self.fc2 = layers.Dense(filters, activation="sigmoid", kernel_initializer=...)

    def call(self, x):
        y = self.pool(x)
        y = self.fc1(y)
        y = self.fc2(y)
        y = tf.reshape(y, (-1, 1, 1, x.shape[-1]))
        return x * y


# 3. Inception Block ( Ib)
class InceptionBlock(layers.Layer):
    def __init__(self, filters):
        super().__init__()
        branch = filters // 4

        self.b1 = ConvBNAct(branch, 1)

        self.b2 = tf.keras.Sequential([
            ConvBNAct(branch, 1),
            ConvBNAct(branch, 3)
        ])

        self.b3 = tf.keras.Sequential([
            ConvBNAct(branch, 1),
            ConvBNAct(branch, 3),
            ConvBNAct(branch, 3)
        ])

        self.b4 = tf.keras.Sequential([
            layers.MaxPool2D(pool_size=3, strides=1, padding="same"),
            ConvBNAct(branch, 1)
        ])

        self.merge = ConvBNAct(filters, 1)

    def call(self, x, training=None):
        p1 = self.b1(x, training=training)
        p2 = self.b2(x, training=training)
        p3 = self.b3(x, training=training)
        p4 = self.b4(x, training=training)
        x = tf.concat([p1, p2, p3, p4], axis=-1)
        return self.merge(x, training=training)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

strategy = tf.distribute.MirroredStrategy()

print("GPUs in sync:", strategy.num_replicas_in_sync)

with strategy.scope():
    inputs = layers.Input(shape=(224, 224, 3))
    


    x = ConvBNAct(64, kernel_size=7, strides=2)(inputs)
    x = layers.MaxPool2D(pool_size=3, strides=2, padding="same")(x)

    shortcut = ConvBNAct(128, kernel_size=1, strides=1)(x)
    x = InceptionBlock(128)(x)
    x = SEBlock(128)(x)
    x = layers.SpatialDropout2D(0.05)(x)
    x = layers.Add()([x, shortcut])

    shortcut = ConvBNAct(256, kernel_size=1, strides=2)(x)
    x = layers.MaxPool2D(pool_size=3, strides=2, padding="same")(x)
    x = InceptionBlock(256)(x)
    x = SEBlock(256)(x)
    x = layers.SpatialDropout2D(0.05)(x)
    x = layers.Add()([x, shortcut])

    # 5. Block 3
    shortcut = ConvBNAct(256, kernel_size=1, strides=2)(x)
    x = layers.MaxPool2D(pool_size=3, strides=2, padding="same")(x)
    x = InceptionBlock(256)(x)
    x = SEBlock(256)(x)
    x = layers.SpatialDropout2D(0.05)(x)
    x = layers.Add()([x, shortcut])

    # 6. Block 4

    shortcut = ConvBNAct(512, kernel_size=1, strides=2)(x) # CHANGED
    x = layers.MaxPool2D(pool_size=3, strides=2, padding="same")(x)
    x = InceptionBlock(512)(x)
    x = SEBlock(512)(x)
    x = layers.SpatialDropout2D(0.05)(x)
    x = layers.Add()([x, shortcut])

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="swish")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.35)(x)

    outputs = layers.Dense(4, activation="softmax")(x)

    model = models.Model(inputs=inputs, outputs=outputs)

In [ ]:
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

with strategy.scope():
    # 1. Define the smooth Cosine Decay schedule
    # steps_per_epoch = len(train_data)
    steps_per_epoch = 350
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=3e-4,
        decay_steps=steps_per_epoch * 100,
        alpha=1e-2
    )

    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=lr_schedule,
        weight_decay=5e-4,
        global_clipnorm=1.0
    )

    # loss = tf.keras.losses.CategoricalCrossentrop(gamma=2.0,label_smoothing=0.1)
    loss = tf.keras.losses.CategoricalFocalCrossentropy(gamma=3.0, label_smoothing=0.1)
    metrics = ["accuracy"]

    model.compile(optimizer=optimizer, loss=loss, metrics=metrics)

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        save_best_only=True,
        monitor="val_accuracy",
        mode="max"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=20,
        restore_best_weights=True
    )
]

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=100,
    callbacks=callbacks
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("Scanning validation images and generating predictions. This will take a few seconds...")

y_true = []
y_pred = []

for images, labels in val_data:
    # true answers
    true_classes = np.argmax(labels.numpy(), axis=1)
    y_true.extend(true_classes)

    # model guesses
    preds = model.predict(images, verbose=0)
    pred_classes = np.argmax(preds, axis=1)
    y_pred.extend(pred_classes)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

class_names = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

plt.figure(figsize=(10, 8))
disp.plot(cmap=plt.cm.Blues, values_format='d', ax=plt.gca())
plt.title('Custom Inception-ResNet-SE: Confusion Matrix', fontsize=16, pad=20)
plt.xlabel('Predicted Tumor Type', fontsize=12)
plt.ylabel('Actual Tumor Type', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.show()

In [ ]:
model.load_weights("best_model.keras")

print("Scanning validation images...")
val_loss, val_accuracy = model.evaluate(val_data)

print("\n" + "="*30)
print(" FINAL VALIDATION RESULTS")
print("="*30)
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")
print("="*30)